# Churn de telecomunicaciones: Modelado y seleccion

Compara regresion logistica sin ponderar y con class_weight balanced. Elige por average precision en validacion; no consulta test.

**Estado:** ejecución completada el 7 de octubre de 2026. Resultados y limitaciones revisados en [metodología](../docs/metodologia.md) y [registro de ejecución](../docs/ejecuciones.md).


In [1]:
from pathlib import Path
import os
import sys
from IPython.display import Markdown, display
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("telecom_churn")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook desde el proyecto, notebooks o su carpeta contenedora.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for folder in ("data/raw", "data/intermediate", "data/processed", "data/output", "models", "reports"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
from telecom_churn.alertas import configurar_notebook, mostrar_alertas
configurar_notebook()


In [2]:
from telecom_churn.flujo import entrenar
display(entrenar(PROJECT_ROOT))

import json
import pandas as pd
from IPython.display import Markdown, display
display(Markdown((PROJECT_ROOT/"reports/modelado/decisiones.md").read_text(encoding="utf-8")))
display(pd.DataFrame(json.loads((PROJECT_ROOT/"reports/modelado/decisiones_validacion.json").read_text(encoding="utf-8"))["decision_candidates"]))

,class_weight,accuracy,average_precision
0,sin ponderar,0.842697,0.356547
1,balanced,0.750936,0.339699


# Calibración y política de decisión

Modelo adicional: **1599 filas para ajuste** y **533 para calibración**. Separación estratificada 75/25 dentro de train; sin ID no se comprueba independencia por persona. Se comparan dos ponderaciones por AP de validación; calibración sigmoid fijada de antemano. El umbral **0.17** minimiza coste ilustrativo FN=5.0, FP=1.0 en la grilla de validación. Se conserva el modelo original para su comparación histórica; las métricas de decisiones corresponden a este ajuste adicional. Validación selecciona modelo y umbral: su rendimiento es optimista y el test queda reservado.


,class_weight,average_precision_validacion
0,None,0.337568
1,balanced,0.332045


## Alertas de la ejecucion

Avisos agrupados con descripcion generica, acciones sugeridas y detalle original. Los errores de celda no se ocultan.

In [3]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/04_modelado.json")
from telecom_churn.informe import generar_informe
generar_informe(PROJECT_ROOT)


WindowsPath('D:/portafolio/Analisis/churn/TelecomChurn/reports/informe_final/informe_final.md')